In [3]:
import os
import huggingface_hub
token = os.getenv('HUGGING_FACE_TOKEN')
huggingface_hub.login(token=token)

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

MODEL_NAME = "meta-llama/Llama-2-7b-chat-hf"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto"
)
print("Modello caricato correttamente!")

tokenizer_config.json:   0%|          | 0.00/1.62k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

...

model-00002-of-00002.safetensors:   0%|          | 0.00/3.50G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/188 [00:00<?, ?B/s]

Some parameters are on the meta device because they were offloaded to the disk.


Modello caricato correttamente!


In [ ]:
prompt = """Analizza il testo seguente e classifica il tipo di errore:
- FORMATO: il testo contiene HTML o caratteri strani
- INCOMPLETO: il testo è troncato
- GRAMMATICA: errori di scrittura evidenti
- DUPLICATO: il testo è simile ad altri articoli
- OK: nessun problema rilevato

Testo:
"{testo}"

Rispondi con una delle categorie sopra.
"""

In [5]:
testo_originale = """
Prosegue lo scontro tra governo e sindacati, ma anche tra sindacati e imprese, sulla riforma del mercato del lavoro.
Il nodo della discussione è la possibile modifica dell'articolo 18.
La norma nello Statuto dei lavoratori del 1970 vieta i licenziamenti in mancanza di giusta causa nelle aziende con più di 15 dipendenti.
Una decisione difficile, una sfida che tanti governi precedenti hanno tentato di vincere, senza successo, e nella quale il governo tecnico
guidato da Mario Monti non intende fallire. Il premier ha reso noto che si andrà avanti nonostante tutto, anche se i sindacati non saranno
daccordo, anche se non si dovesse trovare un accordo unanime. Il mercato e l'Europa chiedono all'Italia un
cambiamento netto nel mercato del lavoro e il terreno di scontro più aspro è quello che riguarda l'articolo 18.
LEGGI ANCHE: Gravidanza e licenziamento
L'articolo 18: Qual è la situazione attuale
Varato nel 1970,  l'articolo 18 afferma che, nelle aziende che hanno più di 15 dipendenti, i lavoratori possono essere licenziati solo per una
giusta causa o un valido motivo.
Se il giudice dovesse valutare che il licenziamento (LEGGI) è stato illeggitimo e in violazione dell'articolo 18 può obbligare
l'azienda al reintegro del lavoratore nella stessa posizione che occupava prima del licenziamento o, se il lavoratore è daccordo,
può liquidarlo con un'indennità pari a 15 mensilità dell'ultimo stipendio o un'indennità crescente con
l'anzianità di servizio.Attualmente quasi due terzi dei lavoratori sono tutelati dall'articolo 18.Cosa cambia con la nuova riforma
L'articolo 18 verrà modificato. Il reintegro sarà obbligatorio solo se il licenziamento è stato discriminatorio,
se l'azienda licenzia per motivi economici sarà obbligata a versare un indennizzo variabile tra 15 e 27 mensilità; se il
licenziamento è causato da motivi disciplinari l'ultima parola spetterà al giudice che potrà decidere se optare per il
reintegro o un indennizzo. Il nuovo articolo 18 si applicherà a tutte le aziende, indipendentemente dal numero di lavoratori.
LEGGI ANCHE: Figli, casa, lavoro, non ce la faccio più.
Le novità per le famiglie e i giovaniLa riforma del lavoro proposta dal governo modifica anche le regole dei contratti a tempo determinato,
che non potranno essere reiterati per più di 36 mesi e sui contratti
a tempo determinato graverà un aumento dei contributi dell'1,4% a carico delle imprese che servirà per finanziare un nuovo sistema
di ammortizzatori sociali. Viene introdotto un nuovo ammortizzatore sociale, chiamato Aspi (assicurazione per l'impiego), che
sostituirà l'assegno di disoccupazione: avrà la durata di un anno per i lavoratori fino a 54 anni, per un massimo di 1.119
euro,è e di 18 mesi per i lavoratori oltre i 54 anni.  Per quanto riguarda le donne saranno vietate le dimissioni in bianco,
un'odiosa pratica messa in atto illegalmente da numerose aziende al momento dell'assunzione, e le aziende verranno invitate a sperimentare
i congedi di paternità obbligatori (LEGGI) per tre anni. Fonte: Liquida
"""

In [6]:
def modify_text(text):
    prompt = f"""Riscrivi il seguente testo correggendo errori, rimuovendo rumore e migliorando la chiarezza, mantenendo il significato originale:

    Testo originale:
    {text}

    Testo corretto:
    """
    
    inputs = tokenizer(prompt, return_tensors="pt").to("mps")  # Usa Metal
    outputs = model.generate(**inputs, max_length = 10_000)
    
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

# Esempio di testo con errori
#testo_originale = "Quasto é un test con errrori di digitazzione e grammattica, da corregggere!"
testo_modificato = modify_text(testo_originale)

print("Testo corretto:", testo_modificato)


/Users/admintemp/miniconda3/envs/ml/lib/python3.11/site-packages/transformers/pytorch_utils.py:335: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  test_elements = torch.tensor(test_elements)


Testo corretto: Riscrivi il seguente testo correggendo errori, rimuovendo rumore e migliorando la chiarezza, mantenendo il significato originale:

    Testo originale:
    
Prosegue lo scontro tra governo e sindacati, ma anche tra sindacati e imprese, sulla riforma del mercato del lavoro.
Il nodo della discussione è la possibile modifica dell'articolo 18.
La norma nello Statuto dei lavoratori del 1970 vieta i licenziamenti in mancanza di giusta causa nelle aziende con più di 15 dipendenti.
Una decisione difficile, una sfida che tanti governi precedenti hanno tentato di vincere, senza successo, e nella quale il governo tecnico
guidato da Mario Monti non intende fallire. Il premier ha reso noto che si andrà avanti nonostante tutto, anche se i sindacati non saranno
daccordo, anche se non si dovesse trovare un accordo unanime. Il mercato e l'Europa chiedono all'Italia un
cambiamento netto nel mercato del lavoro e il terreno di scontro più aspro è quello che riguarda l'articolo 18.
LEGGI AN